# 3D Nuclei Detection with Cellpose
This notebook runs Cellpose on 3D TIFF volumes and extracts centroids + simple time linking.

In [19]:
# Imports (no setup actions requested)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

os.chdir('/home/jschauser/cellTracking/src')

from import_data import load_tiff_files
try:
    from cellpose import models
except ImportError:
    raise ImportError("cellpose not installed. Install with: pip install cellpose[all]")

# Load 3D TIFF volumes
data_folder = os.path.join(os.getcwd(), '../data')
images = load_tiff_files(data_folder)
print(f"Loaded {len(images)} volumes.")



Loaded Embryo_37_intrareg_fuse_t077.tif with shape (790, 2008, 845)
Loaded Embryo_37_intrareg_fuse_t075.tif with shape (790, 2008, 845)
Loaded Embryo_37_intrareg_fuse_t075.tif with shape (790, 2008, 845)
Loaded Embryo_37_intrareg_fuse_t078.tif with shape (790, 2008, 845)
Loaded Embryo_37_intrareg_fuse_t078.tif with shape (790, 2008, 845)
Loaded Embryo_37_intrareg_fuse_t076.tif with shape (790, 2008, 845)
Loaded 4 volumes.
Loaded Embryo_37_intrareg_fuse_t076.tif with shape (790, 2008, 845)
Loaded 4 volumes.


In [ ]:
# Run Cellpose 3D segmentation over all volumes
model_type = 'nuclei'
# Supports only v3 (CellposeModel) APIs
model = models.CellposeModel(gpu=True, model_type=model_type)
channels = [0, 0]  # single-channel grayscale

masks_dict = {}
flows_dict = {}
styles_dict = {}

output_dir = os.path.join('..', 'docs', 'plots', 'cellpose_masks')
os.makedirs(output_dir, exist_ok=True)

for i, (name, vol) in enumerate(images.items()):
    print(f"Segmenting volume {i+1}/{len(images)}: {name} shape={vol.shape}")
    # Ensure volume is float32
    v = vol.astype(np.float32)

    # Specify axes for Cellpose v3 when doing 3D
    if v.ndim == 3:
        # assume TIFF stacks loaded as (Z, Y, X)
        channel_axis = None
        z_axis = 0
    elif v.ndim == 4:
        # common layout (Z, Y, X, C) for multi-channel volumes
        channel_axis = -1
        z_axis = 0
    else:
        raise ValueError(f"Unsupported volume ndim {v.ndim}; expected 3D or 4D.")

    masks, flows, styles = model.eval(
        v,
        channels=channels,
        do_3D=True,
        diameter=None,
        channel_axis=channel_axis,
        z_axis=z_axis,
    )
    masks_dict[name] = masks
    flows_dict[name] = flows
    styles_dict[name] = styles

    # Save mask as TIFF
    base = name.replace('.tif', '')
    mask_path = os.path.join(output_dir, f"{base}_cellpose_mask.tif")
    tifffile.imwrite(mask_path, masks.astype(np.uint16))
    print(f"Saved mask: {mask_path} (labels: {masks.max()})")

print("Segmentation complete.")

model_type argument is not used in v4.0.1+. Ignoring this argument...


Segmenting volume 1/4: Embryo_37_intrareg_fuse_t077.tif shape=(790, 2008, 845)


channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


In [ ]:
# Extract labeled nuclei centroids per volume into a DataFrame
centroids = []
for frame_idx, (name, masks) in enumerate(masks_dict.items()):
    labels = np.unique(masks)
    labels = labels[labels != 0]
    for lab in labels:
        coords = np.argwhere(masks == lab)
        z_mean, y_mean, x_mean = coords.mean(axis=0)
        centroids.append({
            'frame': frame_idx,
            'volume': name.replace('.tif',''),
            'particle': int(lab),
            'z': float(z_mean),
            'y': float(y_mean),
            'x': float(x_mean)
        })

centroids_df = pd.DataFrame(centroids)
print(f"Total nuclei detected: {len(centroids_df)}")
centroids_df.head()

In [ ]:
# Simple nearest-neighbor linking between first two frames (if present)
linked = []
frames_available = sorted(centroids_df['frame'].unique())
if len(frames_available) >= 2:
    f0 = centroids_df[centroids_df.frame == frames_available[0]].copy()
    f1 = centroids_df[centroids_df.frame == frames_available[1]].copy()
    coords0 = f0[['z','y','x']].values
    coords1 = f1[['z','y','x']].values
    for i, c0 in enumerate(coords0):
        dists = np.linalg.norm(coords1 - c0, axis=1)
        j = dists.argmin()
        linked.append({
            'particle_f0': int(f0.iloc[i]['particle']),
            'particle_f1': int(f1.iloc[j]['particle']),
            'dist': float(dists[j])
        })
    links_df = pd.DataFrame(linked)
    print("Linked pairs:")
    display(links_df.head())
else:
    print("Not enough frames for linking.")

In [ ]:
# 3D visualization of centroids for first frame
first_frame = centroids_df[centroids_df.frame == centroids_df.frame.min()]
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(first_frame['x'], first_frame['y'], first_frame['z'], s=12, c='red', alpha=0.6)
ax.set_title('Centroids (Frame 0)')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.show()

In [ ]:
# Save outputs (centroids + links) to CSV
os.makedirs('../checkpoint_data', exist_ok=True)
centroids_out = '../checkpoint_data/cellpose_centroids.csv'
centroids_df.to_csv(centroids_out, index=False)
print('Saved centroids to', centroids_out)

if 'links_df' in globals():
    links_out = '../checkpoint_data/cellpose_links.csv'
    links_df.to_csv(links_out, index=False)
    print('Saved links to', links_out)
else:
    print('No links DataFrame to save.')